In [32]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv'
df = pd.read_csv(url)

df.head()


,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [33]:
#Bigfoot Reports by State

df = df[df['state'].notna()]
df['state'] = df['state'].str.strip().str.title()

state_to_region = {
    'Alabama': 'South', 'Alaska': 'West', 'Arizona': 'West', 'Arkansas': 'South',
    'California': 'West', 'Colorado': 'West', 'Connecticut': 'Northeast', 'Delaware': 'South',
    'Florida': 'South', 'Georgia': 'South', 'Hawaii': 'West', 'Idaho': 'West',
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest',
    'Kentucky': 'South', 'Louisiana': 'South', 'Maine': 'Northeast', 'Maryland': 'South',
    'Massachusetts': 'Northeast', 'Michigan': 'Midwest', 'Minnesota': 'Midwest', 'Mississippi': 'South',
    'Missouri': 'Midwest', 'Montana': 'West', 'Nebraska': 'Midwest', 'Nevada': 'West',
    'New Hampshire': 'Northeast', 'New Jersey': 'Northeast', 'New Mexico': 'West', 'New York': 'Northeast',
    'North Carolina': 'South', 'North Dakota': 'Midwest', 'Ohio': 'Midwest', 'Oklahoma': 'South',
    'Oregon': 'West', 'Pennsylvania': 'Northeast', 'Rhode Island': 'Northeast', 'South Carolina': 'South',
    'South Dakota': 'Midwest', 'Tennessee': 'South', 'Texas': 'South', 'Utah': 'West',
    'Vermont': 'Northeast', 'Virginia': 'South', 'Washington': 'West', 'West Virginia': 'South',
    'Wisconsin': 'Midwest', 'Wyoming': 'West', 'District Of Columbia': 'South'
}

df['region'] = df['state'].map(state_to_region)
df = df[df['region'].notna()]

grouped = df.groupby(['state', 'region']).size().reset_index(name='count')

input_dropdown = alt.binding_select(options=sorted(grouped['region'].unique().tolist()), name='Region:')
selection = alt.selection_point(fields=['region'], bind=input_dropdown, value='West')

chart1 = alt.Chart(grouped).mark_bar().encode(
    x=alt.X('state:N', sort='-y', title='State'),
    y=alt.Y('count:Q', title='Number of Reports'),
    color=alt.Color('region:N', legend=None),
    tooltip=['state:N', 'region:N', 'count:Q']
).add_params(
    selection
).transform_filter(
    selection
).properties(
    title='Bigfoot Sightings by State (Filtered by Region)',
    width=650,
    height=400
)

chart1.save('chart1.html')
chart1


alt.Chart(...)

This bar chart visualizes the number of reported Bigfoot sightings in each U.S. state, grouped and filtered by geographical regions: West, Midwest, South, and Northeast. Each time a region is selected from the dropdown menu, the chart dynamically updates to display only the states within that region and their corresponding sighting counts. The x-axis represents the state names (nominal encoding), and the y-axis shows the number of sightings (quantitative encoding). I used color to distinguish the bars, but disabled the legend to keep the visual minimal since the region context is already provided by the dropdown. The interactivity enables viewers to focus on one region at a time, reducing visual clutter and allowing for easier comparison within regions. On the data processing side, I mapped each state name to its corresponding U.S. Census Bureau-defined region using a Python dictionary, filtered out rows without valid state or region data, and grouped the data by both state and region to count the sightings. This transformation enabled the interactivity to be functional and meaningful. This plot goes beyond default pan/zoom by incorporating a region-level dropdown filter, making the visualization both clearer and more engaging.


In [37]:
#Monthly Bigfoot Report Frequency
df['date'] = pd.to_datetime(df['date'], errors='coerce')

df = df[df['date'].notna()].copy()

df['month'] = df['date'].dt.month

monthly_counts = df.groupby('month').size().reset_index(name='count')

chart2 = alt.Chart(monthly_counts).mark_line(point=True).encode(
    x=alt.X('month:O', title='Month', sort=list(range(1, 13))),
    y=alt.Y('count:Q', title='Number of Reports'),
    tooltip=['month:O', 'count:Q']
).properties(
    title='Monthly Bigfoot Report Frequency',
    width=600,
    height=400
).interactive()

chart2.save("chart2.html")

chart2

alt.Chart(...)

This line chart illustrates the number of Bigfoot reports for each month of the year. The x-axis shows the month (from 1 to 12), and the y-axis represents the count of sightings. I created a new column by extracting the month from the original date field using pd.to_datetime() and .dt.month. The points are connected with a line to show overall trends in sighting frequency across the year. The chart uses ordinal encoding for the month and includes point markers for clarity.

To enhance interactivity, I enabled tooltips that display the number of reports for each point on the line when hovered over. This interactivity allows the viewer to quickly see exact values and adds engagement by allowing users to explore the trend in more detail. From the chart, we can observe that sightings peak in the summer and early fall months, suggesting possible seasonal behavior.

